In [3]:
import os

# Check current working directory
print("Current Directory:", os.getcwd())

# List contents of parent and data directories
print("\nFiles in parent directory (..):", os.listdir('..'))
if os.path.exists('../data'):
    print("Files inside '../data':", os.listdir('../data'))

Current Directory: C:\Users\priya\OneDrive\Desktop\e-commerce customer churn\notebook

Files in parent directory (..): ['dashboard', 'data', 'notebook', 'sql query']
Files inside '../data': ['archive', 'e-commerce_churn.png']


In [5]:
import os

print("Everything inside '../data':")
for root, dirs, files in os.walk('../data'):
    print(f"Directory: {root}")
    print(f"  Folders: {dirs}")
    print(f"  Files: {files}\n")

Everything inside '../data':
Directory: ../data
  Folders: ['archive']
  Files: ['e-commerce_churn.png']

Directory: ../data\archive
  Folders: []
  Files: ['confussion_matrix.png', 'customer_predictor_app.png', 'data_ecommerce_customer_churn.csv', 'final_model.sav', 'logo_shopwise.png']



In [4]:
# 1. View column names and sample data (First 10 rows)
preview_df = duckdb.query("""
    SELECT * 
    FROM ecommerce_churn 
    LIMIT 10
""").df()

# 2. View column data types and non-null counts
print("--- DATASET INFORMATION & TYPES ---")
df_raw.info()

# Display the sample table
print("\n--- FIRST 10 ROWS ---")
preview_df

--- DATASET INFORMATION & TYPES ---
<class 'pandas.DataFrame'>
RangeIndex: 3941 entries, 0 to 3940
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Tenure                    3747 non-null   float64
 1   WarehouseToHome           3772 non-null   float64
 2   NumberOfDeviceRegistered  3941 non-null   int64  
 3   PreferedOrderCat          3941 non-null   str    
 4   SatisfactionScore         3941 non-null   int64  
 5   MaritalStatus             3941 non-null   str    
 6   NumberOfAddress           3941 non-null   int64  
 7   Complain                  3941 non-null   int64  
 8   DaySinceLastOrder         3728 non-null   float64
 9   CashbackAmount            3941 non-null   float64
 10  Churn                     3941 non-null   int64  
dtypes: float64(4), int64(5), str(2)
memory usage: 338.8 KB

--- FIRST 10 ROWS ---


,Tenure,WarehouseToHome,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,DaySinceLastOrder,CashbackAmount,Churn
0,15.0,29.0,4,Laptop & Accessory,3,Single,2,0,7.0,143.32,0
1,7.0,25.0,4,Mobile,1,Married,2,0,7.0,129.29,0
2,27.0,13.0,3,Laptop & Accessory,1,Married,5,0,7.0,168.54,0
3,20.0,25.0,4,Fashion,3,Divorced,7,0,NaN,230.27,0
4,30.0,15.0,4,Others,4,Single,8,0,8.0,322.17,0
5,7.0,16.0,4,Mobile Phone,2,Divorced,2,0,11.0,152.81,0
6,1.0,15.0,6,Mobile Phone,5,Divorced,3,0,2.0,149.51,0
7,1.0,11.0,4,Mobile Phone,5,Single,3,0,1.0,154.73,1
8,11.0,12.0,4,Mobile Phone,3,Married,2,0,4.0,137.02,0
9,17.0,7.0,3,Laptop & Accessory,1,Married,5,1,2.0,157.43,0


In [3]:
import duckdb

# 1. Load the CSV dataset into DuckDB
df_raw = duckdb.query("""
    SELECT * 
    FROM read_csv_auto('../data/archive/data_ecommerce_customer_churn.csv')
""").df()

# 2. Register table in DuckDB
duckdb.register('ecommerce_churn', df_raw)

# 3. Run EDA Overview Query
eda_overview = duckdb.query("""
    SELECT 
        COUNT(*) AS total_customers,
        SUM(Churn) AS total_churned,
        ROUND(AVG(Churn) * 100, 2) AS churn_rate_percent,
        COUNT(*) - COUNT(Tenure) AS missing_tenure,
        COUNT(*) - COUNT(DaySinceLastOrder) AS missing_days_since_last_order
    FROM ecommerce_churn
""").df()

eda_overview

,total_customers,total_churned,churn_rate_percent,missing_tenure,missing_days_since_last_order
0,3941,674.0,17.1,194,213


In [5]:
# 1. Churn rate by Preferred Order Category
category_churn = duckdb.query("""
    SELECT 
        PreferedOrderCat,
        COUNT(*) AS total_customers,
        SUM(Churn) AS churned_customers,
        ROUND(AVG(Churn) * 100, 2) AS churn_rate_pct
    FROM ecommerce_churn
    GROUP BY PreferedOrderCat
    ORDER BY churn_rate_pct DESC
""").df()

# 2. Churn rate by Customer Complaints
complaint_churn = duckdb.query("""
    SELECT 
        Complain,
        COUNT(*) AS total_customers,
        SUM(Churn) AS churned_customers,
        ROUND(AVG(Churn) * 100, 2) AS churn_rate_pct
    FROM ecommerce_churn
    GROUP BY Complain
    ORDER BY Complain DESC
""").df()

print("--- CHURN BY PRODUCT CATEGORY ---")
display(category_churn)

print("\n--- CHURN BY CUSTOMER COMPLAINTS ---")
display(complaint_churn)

--- CHURN BY PRODUCT CATEGORY ---


,PreferedOrderCat,total_customers,churned_customers,churn_rate_pct
0,Mobile Phone,887,255.0,28.75
1,Mobile,559,159.0,28.44
2,Fashion,585,94.0,16.07
3,Laptop & Accessory,1458,142.0,9.74
4,Others,179,12.0,6.70
5,Grocery,273,12.0,4.40



--- CHURN BY CUSTOMER COMPLAINTS ---


,Complain,total_customers,churned_customers,churn_rate_pct
0,1,1113,365.0,32.79
1,0,2828,309.0,10.93


In [6]:
# Churn rate grouped by Tenure ranges
tenure_churn = duckdb.query("""
    SELECT 
        CASE 
            WHEN Tenure IS NULL THEN 'Missing'
            WHEN Tenure <= 6 THEN '0-6 Months'
            WHEN Tenure <= 12 THEN '6-12 Months'
            WHEN Tenure <= 24 THEN '1-2 Years'
            ELSE '2+ Years'
        END AS tenure_group,
        COUNT(*) AS total_customers,
        SUM(Churn) AS churned_customers,
        ROUND(AVG(Churn) * 100, 2) AS churn_rate_pct
    FROM ecommerce_churn
    GROUP BY tenure_group
    ORDER BY churn_rate_pct DESC
""").df()

tenure_churn

,tenure_group,total_customers,churned_customers,churn_rate_pct
0,Missing,194,64.0,32.99
1,0-6 Months,1519,500.0,32.92
2,1-2 Years,1018,61.0,5.99
3,6-12 Months,927,49.0,5.29
4,2+ Years,283,0.0,0.00


In [7]:
# Check missing value overlap and distributions
missing_analysis = duckdb.query("""
    SELECT 
        COUNT(*) AS total_missing_rows,
        SUM(CASE WHEN Tenure IS NULL AND DaySinceLastOrder IS NULL THEN 1 ELSE 0 END) AS missing_both,
        SUM(CASE WHEN Tenure IS NULL THEN 1 ELSE 0 END) AS missing_tenure_only,
        SUM(CASE WHEN DaySinceLastOrder IS NULL THEN 1 ELSE 0 END) AS missing_recency_only
    FROM ecommerce_churn
    WHERE Tenure IS NULL OR DaySinceLastOrder IS NULL
""").df()

missing_analysis

,total_missing_rows,missing_both,missing_tenure_only,missing_recency_only
0,407,0.0,194.0,213.0


In [8]:
# Calculate median values for Tenure and DaySinceLastOrder
medians_df = duckdb.query("""
    SELECT 
        MEDIAN(Tenure) AS median_tenure,
        MEDIAN(DaySinceLastOrder) AS median_recency
    FROM ecommerce_churn
""").df()

medians_df

,median_tenure,median_recency
0,9.0,3.0


In [9]:
# Create a cleaned table with missing values filled using medians
duckdb.query("""
    CREATE OR REPLACE TABLE ecommerce_churn_clean AS
    SELECT 
        * EXCLUDE (Tenure, DaySinceLastOrder),
        COALESCE(Tenure, MEDIAN(Tenure) OVER()) AS Tenure,
        COALESCE(DaySinceLastOrder, MEDIAN(DaySinceLastOrder) OVER()) AS DaySinceLastOrder
    FROM ecommerce_churn
""")

# Verify zero missing values remain
cleaning_verification = duckdb.query("""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(Tenure) AS missing_tenure,
        COUNT(*) - COUNT(DaySinceLastOrder) AS missing_recency
    FROM ecommerce_churn_clean
""").df()

cleaning_verification

,total_rows,missing_tenure,missing_recency
0,3941,0,0


In [10]:
# Save clean dataset to disk
duckdb.query("""
    COPY ecommerce_churn_clean 
    TO '../data/archive/ecommerce_churn_cleaned.csv' 
    (HEADER, DELIMITER ',')
""")
print("Cleaned dataset successfully saved to CSV!")

Cleaned dataset successfully saved to CSV!


In [2]:
import duckdb

# 1. Read the cleaned dataset we exported yesterday
df_clean = duckdb.query("""
    SELECT * 
    FROM read_csv_auto('../data/archive/ecommerce_churn_cleaned.csv')
""").df()

# 2. Register table in DuckDB
duckdb.register('ecommerce_churn_clean', df_clean)

# 3. Run Customer Risk Profiling Query
risk_profiling = duckdb.query("""
    SELECT 
        CASE 
            WHEN Complain = 1 AND Tenure <= 6 AND PreferedOrderCat IN ('Mobile', 'Mobile Phone') THEN 'High Risk'
            WHEN Complain = 1 OR Tenure <= 6 THEN 'Medium Risk'
            ELSE 'Low Risk'
        END AS risk_tier,
        COUNT(*) AS total_customers,
        SUM(Churn) AS actual_churned,
        ROUND(AVG(Churn) * 100, 2) AS actual_churn_rate_pct
    FROM ecommerce_churn_clean
    GROUP BY risk_tier
    ORDER BY actual_churn_rate_pct DESC
""").df()

risk_profiling

,risk_tier,total_customers,actual_churned,actual_churn_rate_pct
0,High Risk,254,172.0,67.72
1,Medium Risk,1913,427.0,22.32
2,Low Risk,1774,75.0,4.23


In [3]:
# Create final dataset with risk_tier included
duckdb.query("""
    CREATE OR REPLACE TABLE ecommerce_churn_final AS
    SELECT 
        *,
        CASE 
            WHEN Complain = 1 AND Tenure <= 6 AND PreferedOrderCat IN ('Mobile', 'Mobile Phone') THEN 'High Risk'
            WHEN Complain = 1 OR Tenure <= 6 THEN 'Medium Risk'
            ELSE 'Low Risk'
        END AS risk_tier
    FROM ecommerce_churn_clean
""")

# Export final enriched dataset for Tableau
duckdb.query("""
    COPY ecommerce_churn_final 
    TO '../data/archive/ecommerce_churn_final.csv' 
    (HEADER, DELIMITER ',')
""")

print("Final dataset exported to ecommerce_churn_final.csv for Tableau!")

Final dataset exported to ecommerce_churn_final.csv for Tableau!
